## Agent Bing Search Sample

* Bing Search APIs retiring on August 11, 2025 https://learn.microsoft.com/en-us/lifecycle/announcements/bing-search-api-retirement
* Display Grounding with Bing Search results https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/tools/bing-grounding?view=azure-python-preview&tabs=python&pivots=overview#how-to-display-grounding-with-bing-search-results


### Setup Foundry Project Client

In [54]:
from typing import Any, Callable, Set, List
import re
import os
import time
import json
from azure.ai.projects import AIProjectClient
from azure.ai.agents.telemetry import trace_function
from azure.ai.agents.models import (
    FunctionTool,
    RequiredFunctionToolCall,
    SubmitToolOutputsAction,
    BingGroundingTool,
    ToolOutput,
    ThreadMessage,
    ToolSet,
    MessageRole,
    MessageTextContent,
    RunStatus,
    RunStep,
)
import azure.ai.agents as agentslib
import azure.ai.projects as projectslib
from opentelemetry import trace
from azure.monitor.opentelemetry import configure_azure_monitor
from dotenv import load_dotenv, find_dotenv

# Your custom Python functions (for "fetch_datetime", etc.)
# from utils.enterprise_functions import enterprise_fns

load_dotenv(dotenv_path=".env", override=True)

from utils.fdyauth import AuthHelper
settings = AuthHelper.load_settings()
credential = AuthHelper.test_credential()

if credential:
    print('Environment and authentication OK')
else:
    print("please login first")

Environment and authentication OK


In [3]:
# new AI Foundry Project resource endpoint / old azure ai services endpoint from the hub/project
project_client = AIProjectClient(
    credential=credential,
    endpoint=settings.project_endpoint,
    # api_version=os.environ["PROJECT_API_VERSION"]
)
print("project_client api version:", project_client._config.api_version)
print(f"azure-ai-agents version: {agentslib.__version__}")
print(f"azure-ai-projects version: {projectslib.__version__}")

project_client api version: 2025-05-15-preview
azure-ai-agents version: 1.2.0b5
azure-ai-projects version: 1.1.0b4


### Set up the helper

* adding custom function 

In [4]:
def print_messages(messages: List[ThreadMessage]) -> None:
    """Helper function to print messages"""
    for message in messages:
        print(f"Role: {message.role}")
        print(f"Content: {message.content[0].text.value}")
        print("-" * 40)

In [5]:
def process_agent_run(thread_id: str, run_id: str, functions: FunctionTool) -> None:
    """Helper function to process an agent run and handle tool calls"""
    run = project_client.agents.runs.get(thread_id=thread_id, run_id=run_id)
    
    # while run.status in ["queued", "in_progress", "requires_action"]:
    while run.status in [RunStatus.QUEUED, RunStatus.IN_PROGRESS, RunStatus.REQUIRES_ACTION]:
        time.sleep(1)
        run = project_client.agents.runs.get(thread_id=thread_id, run_id=run_id)
        if run.status == RunStatus.REQUIRES_ACTION and isinstance(run.required_action, SubmitToolOutputsAction):
            print("Run requires action - processing tool calls")
            tool_calls = run.required_action.submit_tool_outputs.tool_calls
            if not tool_calls:
                print("No tool calls provided - cancelling run")
                project_client.agents.runs.cancel(thread_id=thread_id, run_id=run_id)
                break

            tool_outputs = []
            for tool_call in tool_calls:
                if isinstance(tool_call, RequiredFunctionToolCall):
                    try:
                        print(f"Executing tool_call {tool_call.id} with arguments {tool_call.arguments}")
                        output = functions.execute(tool_call)
                        tool_outputs.append(
                            ToolOutput(tool_call_id=tool_call.id, output=output)
                        )
                        print(f"tool_output for tool_call {tool_call.id}: {output}")
                    except Exception as e:
                        print(f"Error executing tool_call {tool_call.id}: {e}")

            if tool_outputs:
                print(f"Submitting tool outputs: {tool_outputs}")
                project_client.agents.runs.submit_tool_outputs(
                    thread_id=thread_id,
                    run_id=run_id,
                    tool_outputs=tool_outputs
                )

        print(f"Current run status: {run.status}")
    
    return run

In [6]:
def display_message(thread_message: ThreadMessage):
    for agent_message in thread_message.content:
        if isinstance(agent_message, MessageTextContent):
            agent_msg_obj = agent_message.text
            annotations = agent_msg_obj.get("annotations", None)
            print(f"agent message text > {agent_msg_obj.value} (id: {thread_message.id})")
            # get the grounding information from the agent message
            if annotations and isinstance(annotations, list) and len(annotations) > 0:
                # print(f"\nGrounding > {annotations}")
                for annotation in annotations:
                    print(f"{annotation}")

### Setup agent functions

In [7]:
try:
    bing_connection = project_client.connections.get(name=settings.bing_connection_name)
    conn_id = bing_connection.id
    bing_tool = BingGroundingTool(
        connection_id=conn_id,
        count=7, # requested number of search results to return in the response. The default is 5 and themaximum value is 50. The actual number delivered may be less than requested
        set_lang="en", # language to use for user interface strings
        market="en-us", # The market where the results come from.
        # freshness="2025-11-02", # The date or date range used for filtering search results. Single Date
        # freshness="Month",
        freshness="2025-11-01..2025-11-04",
        # * Day: Return webpages that Bing discovered within the last 24 hours.
        # * Week: Return webpages that Bing discovered within the last 7 days.
        # * Month: Return webpages that Bing discovered within the last 30 days. To get articles
        #   discovered by Bing during a specific timeframe, specify a date range in the form:
        #   `YYYY-MM-DD..YYYY-MM-DD`. For example, `freshness=2019-02-01..2019-05-30. To limit the results
        #   to a single date, set this parameter to a specific date. For example, freshness=2019-02-04`.
    )
    print("bing > connected")
except Exception:
    bing_tool = None
    print("bing failed > no connection found or permission issue")

bing > connected


In [8]:
instructions = (
    "You are a helpful enterprise assistant at Contoso. "
    "You have access to following tools. \n\n"
    "## Tools:\n"
    " * bing_grounding: get information about latest news from the public web.\n"
    "\n"
    "## Instructions:\n"
    "You can use the all the tools to answer questions\n"
    "\n"
    "## Guidelines:\n"
    "Provide well-structured and professional answers. Always return the grounding links for citations."
)

AGENT_NAME = "web-search-agent-manual"
found_agent = None
all_agents_list = project_client.agents.list_agents()
for a in all_agents_list:
    if a.name == AGENT_NAME:
        found_agent = a
        break

model_name = settings.model_deployment_name

In [9]:
# Initialize functions
user_functions: Set[Callable[..., Any]] = {}
custom_functions = FunctionTool(functions=user_functions)

toolset = ToolSet()
toolset.add(custom_functions)
toolset.add(bing_tool)

In [10]:
if found_agent:
    # print(found_agent)
    # Update the existing agent to use new tools
    agent = project_client.agents.update_agent(
        agent_id=found_agent.id,
        model=model_name,
        instructions=instructions,
        toolset=toolset,
    )
    print(f"reusing agent > {agent.name} (id: {agent.id})")
else:
    agent = project_client.agents.create_agent(
        model=model_name,
        name=AGENT_NAME,
        instructions=instructions,
        toolset=toolset,
    )
    print(f"creating agent > {agent.name} (id: {agent.id})")

reusing agent > web-search-agent-manual (id: asst_kbt87px6afp7D1swz6KRbaPa)


### Agent run with manual funtion tool calling

In [11]:
USER_QUERY = """what is new about microsoft ai in the news?"""

In [12]:

# Try block content for conversation display
try:
    # Create a thread for the conversation
    thread = project_client.agents.threads.create()
    print(f"Created thread, ID: {thread.id}")

    # User asks about weather with a clear request
    message = project_client.agents.messages.create(
        thread_id=thread.id,
        role=MessageRole.USER,
        content=USER_QUERY,
    )
    print(f"Created initial user message, ID: {message.id}")

    # Start with the weather agent for the initial response
    search_run = project_client.agents.runs.create(
        thread_id=thread.id,
        agent_id=agent.id,
    )
    print(f"Started agent run, ID: {search_run.id}")

    # Process the search agent's run, call the bing tool manually
    search_run = process_agent_run(thread.id, search_run.id, bing_tool)
    print(f"Search agent run completed with status: {search_run.status}")

    # Check messages for Celsius temperatures
    messages = list(project_client.agents.messages.list(thread_id=thread.id))
    print(f"Total messages in thread: {len(messages)}")

    print(f"All messages in the thread: {len(messages)}")
    print_messages(messages)
finally:
    pass    

# finally:
#     # Clean up resources
#     print("\nCleaning up resources...")
#     project_client.agents.delete_agent(agent.id)
#     print("Agents deleted successfully")

Created thread, ID: thread_bs5Hw36l1gcOitgO8eUvgYUw
Created initial user message, ID: msg_4STP1tCSq4zIQYbtHS3sweEg
Started agent run, ID: run_r95w3DVpOxndDuFBCkqzjmtI
Current run status: RunStatus.IN_PROGRESS
Current run status: RunStatus.IN_PROGRESS
Current run status: RunStatus.IN_PROGRESS
Current run status: RunStatus.COMPLETED
Search agent run completed with status: RunStatus.COMPLETED
Total messages in thread: 2
All messages in the thread: 2
Role: MessageRole.AGENT
Content: The latest news about Microsoft AI reveals a significant shift in the company's workforce strategy driven by AI integration. Microsoft plans to grow its headcount again after a period of layoffs, but this growth will be smarter and more leveraged by AI tools. CEO Satya Nadella explained that employees will undergo an "unlearning and learning" process as they adapt to using AI tools such as Microsoft 365 Copilot and GitHub Copilot, both powered by models from OpenAI and Anthropic. These AI capabilities will enab

### Display Grounding with Bing Search results

https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/tools/bing-grounding?view=azure-python-preview&tabs=python&pivots=overview#how-to-display-grounding-with-bing-search-results

According to Grounding with Bing's terms of use and use and display requirements, you need to display both website URLs and Bing search query URLs in your custom interface. 
https://www.microsoft.com/en-us/bing/apis/grounding-legal#use-and-display-requirements

* reconstruct "Bing search query URL"
* get "website URLs" from annotation

In [75]:
def bing_search_query(step: RunStep) -> str | None:
    """Helper function to extract Bing search URL from a run step"""
    if step.type == "tool_calls" and step.step_details and step.step_details.tool_calls and len(step.step_details.tool_calls) > 0:
        # print("Found tool calls in step")
        for tool_call in step.step_details.tool_calls:
            # print(f"tool_call type: {tool_call.type}")
            if tool_call.type == "bing_grounding":
                api_search_query = tool_call.bing_grounding.get("requesturl", "")
                query_parts = api_search_query.split('=')
                user_query = query_parts[-1].replace(' ','%20')
                url_prefix = query_parts[0].replace('api.bing.microsoft.com/v7.0', 'www.bing.com')
                # replace the api.bing.microsoft.com/v7.0 with www.bing.com
                bing_search_query = url_prefix + '=' + user_query
                return bing_search_query
    return None

In [76]:
# reconstruct bing search query
run_steps = project_client.agents.run_steps.list(thread_id=thread.id, run_id=search_run.id)

for step in run_steps:
    query_url = bing_search_query(step)
    if query_url:
        print(f"Bing Search URL: {query_url}")

Bing Search URL: https://www.bing.com/search?q=latest%20news%20about%20Microsoft%20AI%20November%202025


In [ ]:
agent_message_annotation: ThreadMessage = project_client.agents.messages.get_last_message_by_role(thread_id = thread.id, role=MessageRole.AGENT)

# display the grounding information from the agent message, website urls
display_message(agent_message_annotation)

agent message text > Recent news about Microsoft AI highlights several important developments:

1. Microsoft plans to resume hiring after multiple rounds of layoffs, but with a strategic focus on leveraging AI to enhance productivity. CEO Satya Nadella explained that future headcount growth will be "smarter and more leveraged," with employees adopting AI tools like Microsoft 365 Copilot and GitHub Copilot to transform how work is done. This hiring approach emphasizes smaller teams equipped with AI capabilities to achieve much more than before. Employees will undergo a process of "unlearning and learning" to effectively integrate AI into their workflows.【3:0†source】【3:1†source】【3:4†source】【3:5†source】

2. Microsoft is significantly increasing its investment in AI infrastructure, with capital expenditures in the billions focused on cloud infrastructure and AI hardware such as GPUs and CPUs. This investment positions Microsoft as a leader in AI cloud services and supports its growing port